In [ ]:
import json
import pennylane as qml
import pennylane.numpy as np

class AbsMagnetization(qml.measurements.StateMeasurement):
    """A measurement class that estimates <|M|>."""

    def process_state(self, state, wire_order):
        """Calculates <|M|>.

        Args:
            state (Sequence[complex]): quantum state with a flat shape. It may also have an
                optional batch dimension.

            wire_order (Wires): wires determining the subspace that the state acts on; a matrix of
                dimension 2**n that acts on a subspace of n wires
        
        Returns: 
            abs_mag (float): <|M|>

        
        See the docs for more information:
        https://docs.pennylane.ai/en/stable/code/api/pennylane.measurements.StateMeasurement.html
        """

        num_qubits = len(wire_order)
        abs_mag_sum_numerator = 0.0 # This will accumulate sum_sigma |<sigma|psi>|^2 * |sum_i <sigma|Z_i|sigma>|

        # Iterate through all computational basis states |sigma>
        for k in range(2**num_qubits):
            # Calculate sum_{i=1}^N <sigma | Z_i | sigma>
            # For a basis state |sigma>, Z_i acts as +1 if the i-th bit is 0 (spin up),
            # and -1 if the i-th bit is 1 (spin down).
            # The sum is (number of 0s) - (number of 1s)
            # This is equivalent to num_qubits - 2 * (number of 1s)
            
            # Count the number of '1's in the binary representation of k
            num_ones = bin(k).count('1')
            
            # Calculate the sum of Z_i expectation values for this specific basis state |sigma>
            sum_Z_k_in_basis = num_qubits - 2 * num_ones
            
            # Add to the total sum, weighted by the probability of measuring |sigma> and the absolute value
            abs_mag_sum_numerator += np.abs(state[k])**2 * np.abs(sum_Z_k_in_basis)
        
        # According to the definition <|M|> = (1/N) * sum_sigma ...
        return abs_mag_sum_numerator / num_qubits


def tfim_ground_state(num_qubits, h):
    """Calculates the ground state of the 1D TFIM Hamiltonian.

    Args:
        num_qubits (int): The number of qubits / spins.
        h (float): The transverse field strength.

    Returns:
        (numpy.tensor): The ground state.
    """

    coeffs = []
    ops = []

    # Nearest-neighbour Z_i Z_{i+1} terms for a 1D chain
    for i in range(num_qubits - 1):
        coeffs.append(-1.0) # Coefficient for -Z_i Z_{i+1}
        ops.append(qml.PauliZ(i) @ qml.PauliZ(i+1))

    # Transverse field -h X_i terms
    for i in range(num_qubits):
        coeffs.append(-h) # Coefficient for -h X_i
        ops.append(qml.PauliX(i))
    
    # Construct the Hamiltonian
    H = qml.Hamiltonian(coeffs, ops)

    # Get the matrix representation of the Hamiltonian
    # Wires must be explicitly provided to qml.matrix for the correct ordering
    H_matrix = qml.matrix(H, wires=range(num_qubits))

    # Calculate eigenvalues and eigenvectors
    # np.linalg.eigh returns eigenvalues in ascending order, and corresponding eigenvectors as columns
    eigenvalues, eigenvectors = np.linalg.eigh(H_matrix)

    # The ground state is the eigenvector corresponding to the smallest eigenvalue (first column)
    ground_state = eigenvectors[:, 0]

    return ground_state


dev = qml.device("default.qubit")


@qml.qnode(dev)
def magnetization(num_qubits, h):
    """Calculates the absolute value of the magnetization of the 1D TFIM
    Hamiltonian.

    Args:
        num_qubits (int): The number of qubits / spins.
        h (float): The transverse field strength.

    Returns:
        (float): <|M|>.
    """

    # Get the ground state of the TFIM Hamiltonian for given parameters
    ground_state = tfim_ground_state(num_qubits, h)

    # Prepare the quantum circuit in the calculated ground state
    qml.StatePrep(ground_state, wires=list(range(num_qubits)))

    # Return the custom absolute magnetization measurement
    # This measurement will process the prepared state to calculate <|M|>
    return AbsMagnetization(wires=list(range(num_qubits)))


def critical_point_estimate(mags, h_values):
    """Provides a finite-size estimate of the critical point of the 1D TFIM
    Hamiltonian. The estimate is done by taking the average value of h for which
    adjacent values of <|M|> differ the most.

    Args:
        mags (numpy.tensor):
            <|M|> values for various values of h (the transverse field strength).
        h_values (numpy.tensor): The transverse field strength values.

    Returns:
        (float): The critical point estimate, h_c.
    """
    differences = [np.abs(mags[i] - mags[i + 1]) for i in range(len(mags) - 1)]
    ind = np.argmax(np.array(differences))

    h_c = np.mean([h_values[ind], h_values[ind + 1]])
    return h_c


# These functions are responsible for testing the solution.
def run(test_case_input: str) -> str:
    num_qubits = json.loads(test_case_input)
    h_values = np.arange(0.2, 1.1, 0.005)
    mags = []

    for h in h_values:
        mags.append(magnetization(num_qubits, h) / num_qubits)

    output = critical_point_estimate(np.array(mags), h_values)

    return str(output)


def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)

    assert np.isclose(solution_output, expected_output, rtol=5e-3)


# These are the public test cases
test_cases = [
    ('5', '0.6735'),
    ('2', '0.3535')
]

# This will run the public test cases locally
for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        if message := check(output, expected_output):
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")

        else:
            print("Correct!")